# Module 3 — Add AgentCore Memory

In Module 2 you deployed the Chief of Staff agent to AgentCore Runtime. It works — but it's
**stateless**: every invocation is a fresh, isolated session, so the agent forgets you the moment
a call ends. A real chief of staff remembers what you discussed last week.

In this module you give the **same** deployed agent **cross-session memory** using **Amazon Bedrock
AgentCore Memory** — a managed service for agent memory. You'll keep `build_agent_options()` as the
**one source of truth** (no agent rewrite); memory is layered on in the thin entrypoint.

> **Scope:** this module is **single-tenant** — one fixed user (`techstart-cos`) — to keep the focus
> on *how memory works*. Production multi-tenant memory (per-user isolation, identity from inbound
> auth) is a deliberate next step, not covered here.

### What you'll do

| Step | What happens |
|------|--------------|
| **1. The two layers** | Short-term events vs. long-term extraction — and why it matters for the demo |
| **2. The memory layer** | A thin `memory/session.py` the entrypoint calls around each turn — agent unchanged |
| **3. Configure** | Add a memory resource to `agentcore.json` (`memories[]`) |
| **4. Deploy** | `agentcore deploy` provisions the runtime **and** the memory, and auto-wires IAM |
| **5. Session 1** | Tell the agent something only *this* conversation knows |
| **6. Session 2** | A brand-new session **recalls** it — the cross-session "aha" |
| **7. Prove it's memory** | Same prompt, memory **off** → it forgets. Honest A/B. |
| **8. Long-term recall** | Inspect the facts/preferences the service **extracted** in the background |
| **9. Clean up** | Tear down the runtime **and** the memory resource |


## Setup

Run the cell below to install all dependencies (Node.js, AgentCore CLI, Python packages) and
register the Jupyter kernel. After it completes, **select the `module-3-memory` kernel** from the
kernel picker (top-right) and continue with the rest of the notebook.

In [ ]:
!bash setup.sh

In [1]:
!curl -fsSL https://rpm.nodesource.com/setup_20.x | sudo bash -
!sudo yum install -y nodejs-20.20.2
!sudo npm install -g @aws/agentcore@0.17.0
!cd agentcore/cdk && npm ci
!agentcore --version

2026-06-06 07:29:41 - Cleaning up old repositories...
2026-06-06 07:29:41 - Old repositories removed
2026-06-06 07:29:41 - Supported architecture: aarch64
2026-06-06 07:29:41 - Added N|Solid repository for LTS version: 20.x
2026-06-06 07:29:41 - dnf available, updating...
Node.js Packages for Linux RPM based distros -  181 kB/s | 3.0 kB     00:00    
Metadata cache created.
N|Solid Packages for Linux RPM based distros -  172 kB/s | 3.0 kB     00:00    
Metadata cache created.
2026-06-06 07:29:41 - Repository is configured and updated.
2026-06-06 07:29:41 - You can use N|solid Runtime as a node.js alternative
2026-06-06 07:29:41 - To install N|solid Runtime, run: dnf install nsolid -y
2026-06-06 07:29:41 - Run 'dnf install nodejs -y' to complete the installation.
Last metadata expiration check: 0:00:01 ago on Sat 06 Jun 2026 07:29:41 AM UTC.
Package nodejs-2:20.20.2-1nodesource.aarch64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹

### Install dependencies & register kernel

In [2]:
import subprocess, shutil
from pathlib import Path

module_dir = Path.cwd()
module_name = module_dir.name  # e.g. "module-3-memory"

# 1) .env
if not (module_dir / ".env").exists():
    shutil.copy(".env.example", ".env")
    print("✅ created .env from .env.example — edit it if you need a different model/region")
else:
    print("✅ .env already exists")

# 2) uv sync
subprocess.run(["uv", "sync"], check=True, cwd=module_dir)
print("✅ uv sync done")

# 3) Register Jupyter kernel
venv_python = module_dir / ".venv" / "bin" / "python"
subprocess.run([
    str(venv_python), "-m", "ipykernel", "install",
    "--user", "--name", module_name, "--display-name", module_name,
], check=True)
print(f"✅ Kernel registered: {module_name}")

✅ created .env from .env.example — edit it if you need a different model/region


Using CPython 3.11.14
Creating virtual environment at: .venv
Resolved 164 packages in 207ms
Installed 142 packages in 255ms
 + annotated-types==0.7.0
 + anyio==4.13.0
 + asgiref==3.11.1
 + asttokens==3.0.1
 + attrs==26.1.0
 + aws-opentelemetry-distro==0.17.1
 + bedrock-agentcore==1.13.0
 + boto3==1.43.21
 + botocore==1.43.24
 + cachetools==6.2.4
 + certifi==2026.5.20
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + claude-agent-sdk==0.2.88
 + click==8.4.1
 + comm==0.2.3
 + cryptography==48.0.0
 + debugpy==1.8.21
 + decorator==5.3.1
 + executing==2.2.1
 + googleapis-common-protos==1.75.0
 + grpcio==1.81.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + httpx-sse==0.4.3
 + idna==3.18
 + importlib-metadata==8.7.1
 + ipykernel==7.2.0
 + ipython==9.14.1
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jmespath==1.1.0
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + jupyter-client==8.9.0
 + jupyter-core==5.9.1
 + matplotlib-inline==0.2.2
 + mcp==1.27.2
 + nest-asyncio=

✅ uv sync done
Installed kernelspec module-3-memory in /home/participant/.local/share/jupyter/kernels/module-3-memory
✅ Kernel registered: module-3-memory


### Generate deployment target (`aws-targets.json`)

In [1]:
import json, os, subprocess

# --- Resolve region: single source of truth for the whole notebook ---
# Priority: AWS_REGION env (set by Workshop Studio) → AWS CLI config → fallback
_cli_region = subprocess.run(
    ["aws", "configure", "get", "region"], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get("AWS_REGION") or _cli_region or "us-west-2"

account_id = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
    capture_output=True, text=True,
).stdout.strip()

# Generate aws-targets.json from the resolved values
targets = [
    {
        "name": "default",
        "description": "Workshop deployment target (auto-generated).",
        "account": account_id,
        "region": REGION,
    }
]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)

print(f"✅ agentcore/aws-targets.json written:")
print(f"   account: {account_id}")
print(f"   region:  {REGION}")

✅ agentcore/aws-targets.json written:
   account: 361691913159
   region:  us-east-1


## Step 1 — Two layers of memory (and the one gotcha that shapes the demo)

AgentCore Memory has two layers, and Module 3 uses **both**:

| Layer | API | What it holds | When it's available |
|---|---|---|---|
| **Short-term** | `create_event` / `list_events` | Raw conversation turns, per `(actor, session)` | **Immediately** |
| **Long-term** | extraction → `retrieve_memory_records` | Distilled **facts** & **preferences**, per actor | **After async extraction (~1–2 min)** |

The crucial gotcha: **long-term extraction is asynchronous.** When you write a turn, the service
extracts facts/preferences in the background — in our us-west-2 measurement, **preferences appeared
~64s and facts ~80s later** (and the memory itself takes ~150s to become `ACTIVE` on first create).

So a fact you state in one breath is **not** semantically retrievable the next second. That's why:

- **Live cross-session recall rides on the SHORT-TERM layer** (`list_events` of the actor's prior
  sessions) — instant and reliable.
- **The long-term layer is the "it learned about you over time" beat** — we inspect it after a
  natural gap (Step 8), where the extraction delay doesn't matter.

Designing the demo around this is what makes it work live instead of mysteriously returning nothing.


## Step 2 — The memory layer (the agent stays the source of truth)

We do **not** rewrite the agent. `build_agent_options()` in `agent.py` is still the one source of
truth. Module 3 adds exactly two thin pieces:

1. **`chief_of_staff_agent/memory/session.py`** — wraps the AgentCore Memory data plane behind two
   methods: `retrieve_context(query)` (turn start) and `record_turn(user, assistant)` (turn end).
2. **A few lines in `agent_agentcore.py`** — recall before the turn, inject via the additive
   `system_prompt_suffix`, record after the turn.

Run the cell to confirm the entrypoint still reuses `build_agent_options()` and injects memory
through the suffix seam (never a forked system prompt):


In [2]:
import sys
sys.path.insert(0, "chief_of_staff_agent")

import inspect
import agent_agentcore

src = inspect.getsource(agent_agentcore)
# Reuses the shared identity builder...
assert "from agent import build_agent_options" in src
# ...injects memory via the additive suffix (NOT a forked system_prompt=)...
assert "system_prompt_suffix=" in src and "system_prompt=" not in src
# ...and takes a 2nd `context` param so the runtime delivers the session id.
params = list(inspect.signature(agent_agentcore.invoke).parameters)
assert params[:2] == ["payload", "context"], params
print("✅ entrypoint reuses build_agent_options(), injects memory via system_prompt_suffix")
print("   invoke signature:", params)

✅ entrypoint reuses build_agent_options(), injects memory via system_prompt_suffix
   invoke signature: ['payload', 'context']


Here's the shape of the entrypoint — note it's still *plumbing*, not agent logic:

```python
@app.entrypoint
async def invoke(payload, context):                    # ← 2nd param `context` for session id
    prompt     = (payload or {}).get("prompt")
    memory_on  = (payload or {}).get("memory", True)   # ← A/B toggle for the demo
    actor_id   = (payload or {}).get("actor_id") or "techstart-cos"   # single-tenant
    session_id = getattr(context, "session_id", None)

    mem = get_memory(actor_id, session_id)
    suffix = mem.retrieve_context(prompt) if memory_on else ""    # ← recall (turn start)
    options = build_agent_options(system_prompt_suffix=suffix)    # ← same identity + memory

    chunks = []
    async with ClaudeSDKClient(options=options) as agent:
        await agent.query(prompt)
        async for msg in agent.receive_response():
            for block in getattr(msg, "content", []) or []:
                if getattr(block, "text", None):
                    chunks.append(block.text); yield block.text
    if memory_on:
        mem.record_turn(prompt, "".join(chunks))         # ← persist (turn end)
```

`get_memory()` degrades gracefully: if the memory resource isn't configured (e.g. local
`agentcore dev`, where Memory isn't available) it returns a no-op and the agent simply behaves like
Module 2 — stateless, never broken.


## Step 3 — Configure the memory resource

Memory is declared in `agentcore/agentcore.json` under `memories[]` (already set up). It defines
**one** memory with the two long-term strategies and **actor-scoped namespaces**:

```json
"memories": [{
  "name": "CosMemory",
  "eventExpiryDuration": 7,
  "strategies": [
    { "type": "SEMANTIC",        "name": "facts", "namespaces": ["users/{actorId}/facts"] },
    { "type": "USER_PREFERENCE", "name": "prefs", "namespaces": ["users/{actorId}/preferences"] }
  ]
}]
```

- **SEMANTIC** extracts durable *facts* ("the raise target is $X"); **USER_PREFERENCE** captures
  *how the user likes things* ("report runway in weeks").
- `eventExpiryDuration: 7` caps storage (raw events expire after 7 days — the minimum — which keeps
  workshop cost tiny).
- The `{actorId}` placeholder scopes records per user. (Single-tenant here = one actor, but the
  template is how you'd grow to multi-tenant later.)


Then validate:


In [3]:
import subprocess, json
print(subprocess.run(["agentcore", "validate"], capture_output=True, text=True).stdout)

cfg = json.load(open("agentcore/agentcore.json"))
print("memories:", json.dumps(cfg["memories"], indent=2))

Valid

memories: [
  {
    "name": "CosMemory",
    "eventExpiryDuration": 7,
    "strategies": [
      {
        "type": "SEMANTIC",
        "name": "facts",
        "namespaces": [
          "users/{actorId}/facts"
        ]
      },
      {
        "type": "USER_PREFERENCE",
        "name": "prefs",
        "namespaces": [
          "users/{actorId}/preferences"
        ]
      }
    ]
  }
]


## Step 4 — Deploy (provisions runtime **and** memory)

`agentcore deploy` now provisions **two** things: the agent runtime (as in Module 2) *and* the
`CosMemory` resource. It also **auto-wires the memory IAM** onto the runtime's execution role
(`CreateEvent`, `ListEvents`, `ListSessions`, `RetrieveMemoryRecords`, …) and **injects the memory id**
as the env var `MEMORY_COSMEMORY_ID` — which `memory/session.py` reads. You don't write any IAM.

> **Local note:** Memory is **not** available during `agentcore dev` (the service is cloud-only). So
> unlike Module 2, the dev loop here is: **deploy → invoke → observe**.

Run in a **terminal** (creates real resources; takes a few minutes; the memory takes ~2–3 min to go
`ACTIVE`):

```bash
agentcore deploy -y
```

Check status (the runtime and the memory both appear):


In [6]:
!agentcore deploy -y

✓ Load deployment target
⠋ Validate project...(node:372505) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate project
✓ Build CDK project...
✓ Synthesize CloudFormation...
✓ Check bootstrap status...
✓ Check stack status...
⠋ Deploy to AWS...(node:372505) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Deploy to AWS
✓ Persi

In [7]:
import subprocess
print(subprocess.run(["agentcore", "status"], capture_output=True, text=True).stdout or "run `agentcore deploy -y` in a terminal first")

AgentCore Status (target: default, us-west-2)

Agents
  cos: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-west-2:3616919131
59:runtime/cosmemory_cos-PN58aI5sXU)
  URL: https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-west-2%3A361691913159%3Aruntime%2Fcosmemory_cos-PN58aI5sXU/
invocations

Memories
  CosMemory: Deployed (SEMANTIC, USER_PREFERENCE) (arn:aws:bedrock-agentcore:us-
west-2:361691913159:memory/cosmemory_CosMemory-xK1cg1E4VJ)



We'll invoke with **boto3** directly so we control the exact payload — the `runtimeSessionId`
(which scopes short-term memory) and the optional `"memory"` flag (for the A/B in Step 7). This
helper resolves the deployed runtime ARN from `agentcore status` and calls `invoke_agent_runtime`:


> The plain `agentcore invoke --session-id <id> "<prompt>"` CLI also works for a quick check (it sends `{"prompt": ...}`). We use boto3 here only so we can add the `"memory"` flag and pin the `runtimeSessionId` precisely.

In [15]:
import boto3, json, subprocess, re

REGION = json.load(open("agentcore/aws-targets.json"))[0]["region"]
_ANSI = re.compile(r"\x1b\[[0-9;?]*[a-zA-Z]")

def _runtime_arn():
    out = subprocess.run(["agentcore", "status", "--json"], capture_output=True, text=True).stdout
    clean = _ANSI.sub("", out)
    data = json.loads(clean)
    for r in data.get("resources", []):
        if r.get("resourceType") == "agent":
            return r["identifier"]
    raise RuntimeError("Could not find runtime ARN — is the agent deployed? Run `agentcore deploy -y`.")

_client = boto3.client("bedrock-agentcore", region_name=REGION)

def invoke(prompt, session_id, memory=True):
    """Invoke the deployed agent with an explicit session id and memory flag."""
    payload = {"prompt": prompt, "memory": memory}
    resp = _client.invoke_agent_runtime(
        agentRuntimeArn=_runtime_arn(),
        runtimeSessionId=session_id,                       # >=33 chars; scopes short-term memory
        payload=json.dumps(payload).encode(),
        contentType="application/json", accept="application/json",
    )
    body = resp["response"].read() if hasattr(resp["response"], "read") else resp["response"]
    text = body.decode() if isinstance(body, (bytes, bytearray)) else str(body)
    print(text)
    return text

print(f"Ready to invoke in {REGION}.")

Ready to invoke in us-west-2.


## Step 5 — Session 1: tell the agent something only this conversation knows

We deliberately state a fact that is **not** in the company `CLAUDE.md` (which hardcodes a $30M Series
B). We invent **$42.5M** live — so if a later session repeats it, that can only be **memory**, not the
always-on company context.

Session ids must be **≥33 characters** (an AgentCore requirement — shorter ids are silently dropped).


In [16]:
SESSION_A = "sess-m3-monday-planning-000000000001"   # >=33 chars
ACTOR = "techstart-cos"

invoke(
    "Note for the record: we're modeling our Series B raise at a $42.5M target with an 18-month "
    "bridge. And going forward, always report runway in weeks, not months.",
    session_id=SESSION_A,
)
# Behind the scenes: record_turn() wrote this exchange to short-term memory and queued the
# long-term extraction (a SEMANTIC fact + a USER_PREFERENCE).

data: "Noted on both points:\n\n1. **Series B parameters updated**: $42.5M target raise with an 18-month bridge runway assumption. That's a meaningful step up from the earlier $30M figure — likely reflecting the ARR trajectory and market positioning for a stronger negotiating stance.\n\n2. **Runway reporting**: Going forward, I'll report runway in **weeks** rather than months. For reference, our current position translates to approximately **87 weeks** of runway (based on $10M cash / ~$500K monthly burn ≈ 20 months × 4.33 weeks).\n\nI'll apply both of these in all future financial analyses and board-facing materials."




'data: "Noted on both points:\\n\\n1. **Series B parameters updated**: $42.5M target raise with an 18-month bridge runway assumption. That\'s a meaningful step up from the earlier $30M figure — likely reflecting the ARR trajectory and market positioning for a stronger negotiating stance.\\n\\n2. **Runway reporting**: Going forward, I\'ll report runway in **weeks** rather than months. For reference, our current position translates to approximately **87 weeks** of runway (based on $10M cash / ~$500K monthly burn ≈ 20 months × 4.33 weeks).\\n\\nI\'ll apply both of these in all future financial analyses and board-facing materials."\n\n'

## Step 6 — Session 2: a brand-new session **recalls** it (the cross-session "aha")

Now open a **different** session for the **same** actor — as if you came back the next day. It shares
no transcript with Session 1. Yet the agent recalls the $42.5M target, because `retrieve_context()`
pulled the actor's prior-session turns from **short-term** memory (instant — no extraction wait):


In [17]:
SESSION_B = "sess-m3-tuesday-followup-00000000002"   # different session, same actor

invoke(
    "What raise target did we land on for the Series B, and how should I report runway?",
    session_id=SESSION_B,
)
# Expect: ~$42.5M and "in weeks" — recalled from Session 1, not from CLAUDE.md.

data: "Two items from our standing decisions:\n\n1. **Series B target**: We're modeling at **$42.5M** with an **18-month bridge** runway assumption. (This supersedes the earlier $30M placeholder in the Q2 priorities doc.)\n\n2. **Runway reporting**: Always in **weeks**, not months. For current reference, our position is approximately **87 weeks** (~$10M cash ÷ ~$500K monthly burn ≈ 20 months × 4.33 ≈ 87 weeks).\n\nLet me know if you'd like me to run updated forecasts against the $42.5M target or prep anything for board materials."




'data: "Two items from our standing decisions:\\n\\n1. **Series B target**: We\'re modeling at **$42.5M** with an **18-month bridge** runway assumption. (This supersedes the earlier $30M placeholder in the Q2 priorities doc.)\\n\\n2. **Runway reporting**: Always in **weeks**, not months. For current reference, our position is approximately **87 weeks** (~$10M cash ÷ ~$500K monthly burn ≈ 20 months × 4.33 ≈ 87 weeks).\\n\\nLet me know if you\'d like me to run updated forecasts against the $42.5M target or prep anything for board materials."\n\n'

That's the whole point of Module 3: a **new session**, no shared transcript, and the agent still
remembers — because memory now spans sessions.


## Step 7 — Prove it's memory, not the prompt (honest A/B)

A skeptic could ask: "Did it really *remember*, or did you just word the prompt to elicit that?" So we
run the **same prompt on the same deployment** with memory turned **off** (`memory=False`). Same code,
same prompt — the only difference is whether memory is consulted:


In [18]:
invoke(
    "What raise target did we land on for the Series B, and how should I report runway?",
    session_id="sess-m3-control-memory-off-0000000003",
    memory=False,                                  # ← skip retrieval + recording for this turn
)
# Expect: it does NOT know $42.5M — it asks what target to model. Memory off → no recall.

data: "Based on the company context I have, here's what I can tell you:\n\n## Series B Target\n\nThe target we set for Series B conversations is **$30M**. This was established as a Q2 2024 priority when we began those conversations.\n\n## Runway Reporting — Important Note\n\nThere's a significant discrepancy I need to flag. Our documented figures show:\n\n- **Cash in Bank**: $10M\n- **Monthly Burn**: ~$500K\n- **Runway**: 20 months (until September 2025)\n\nHowever, today is **June 2026** — that's 9 months *past* the projected runway end date. This means either:\n\n1. The Series B closed and our cash position has changed materially, or\n2. Burn rate decreased, or\n3. Revenue growth offset burn significantly (ARR was $2.4M and growing 15% MoM)\n\nLet me check if there's updated financial data on file:"

data: "## Here's the full picture:\n\n### Series B Target\n**$30M raise** — that's the figure in our planning documents.\n\n### How to Report Runway\n\nBased on the most recent financial

'data: "Based on the company context I have, here\'s what I can tell you:\\n\\n## Series B Target\\n\\nThe target we set for Series B conversations is **$30M**. This was established as a Q2 2024 priority when we began those conversations.\\n\\n## Runway Reporting — Important Note\\n\\nThere\'s a significant discrepancy I need to flag. Our documented figures show:\\n\\n- **Cash in Bank**: $10M\\n- **Monthly Burn**: ~$500K\\n- **Runway**: 20 months (until September 2025)\\n\\nHowever, today is **June 2026** — that\'s 9 months *past* the projected runway end date. This means either:\\n\\n1. The Series B closed and our cash position has changed materially, or\\n2. Burn rate decreased, or\\n3. Revenue growth offset burn significantly (ARR was $2.4M and growing 15% MoM)\\n\\nLet me check if there\'s updated financial data on file:"\n\ndata: "## Here\'s the full picture:\\n\\n### Series B Target\\n**$30M raise** — that\'s the figure in our planning documents.\\n\\n### How to Report Runway\\n\

Side by side: **memory on** → recalls $42.5M; **memory off** → forgets. Same agent, same prompt.
That's the honest proof the recall comes from AgentCore Memory.


## Step 8 — Long-term memory: what the service **extracted**

Short-term memory replays raw turns. The **long-term** layer is different: the service has been
distilling your conversations into durable **facts** and **preferences** in the background. By now
(a few minutes after Session 1) extraction has run, so we can retrieve them semantically.

This is the "the agent *learned* about you" layer — and inspecting it makes the two-layer model
concrete:


In [20]:
import json

_status_out = subprocess.run(["agentcore", "status", "--json"], capture_output=True, text=True).stdout
_status_data = json.loads(_ANSI.sub("", _status_out))
MEMORY_ID = next(
    (r["identifier"].rsplit("/", 1)[-1] for r in _status_data.get("resources", [])
     if r.get("resourceType") == "memory"),
    None,
)
print("memory id:", MEMORY_ID or "(deploy first)")

for ns_label, ns in [("FACTS", f"users/{ACTOR}/facts"), ("PREFERENCES", f"users/{ACTOR}/preferences")]:
    try:
        r = _client.retrieve_memory_records(
            memoryId=MEMORY_ID, namespace=ns,
            searchCriteria={"searchQuery": "Series B raise target and runway reporting", "topK": 5},
        )
        print(f"\n{ns_label} ({ns}):")
        for rec in r.get("memoryRecordSummaries", []):
            print("  •", rec["content"]["text"], " (score:", round(rec.get("score", 0), 3), ")")
    except Exception as e:
        print(f"\n{ns_label}: not ready yet ({type(e).__name__}). Long-term extraction is async "
              f"(~1–2 min) — re-run this cell shortly.")

memory id: cosmemory_CosMemory-xK1cg1E4VJ

FACTS (users/techstart-cos/facts):
  • The user's company is targeting a Series B raise of $42.5M with an 18-month bridge runway assumption (as of 2026-06-06), superseding an earlier $30M placeholder in the Q2 priorities doc.  (score: 0.528 )
  • The user's company has approximately $10M cash and ~$500K monthly burn, giving roughly 87 weeks of runway (as of 2026-06-06).  (score: 0.362 )
  • The user prefers runway to always be reported in weeks, not months.  (score: 0.356 )

PREFERENCES (users/techstart-cos/preferences):
  • {"context":"The user explicitly stated the Series B raise is modeled at a $42.5M target with an 18-month bridge.","preference":"Series B raise modeled at a $42.5M target with an 18-month bridge","categories":["finance","startup","fundraising"]}  (score: 0.499 )
  • {"context":"The user explicitly stated that financial runway should always be reported in weeks rather than months going forward.","preference":"Prefer runway t

> If a namespace is empty, extraction simply hasn't finished — re-run the cell in a minute. This
> async delay is exactly why **Steps 5–6 rode on short-term memory** for the live recall.


## Step 9 — Clean up

Tear down **both** the runtime and the memory resource so nothing keeps billing. Run in a **terminal**:

```bash
agentcore remove agent  --name cos          # remove the runtime from config
agentcore remove memory --name CosMemory    # remove the memory from config
agentcore deploy -y                         # apply removals → destroys both
```

Confirm nothing lingers:


In [ ]:
import subprocess
print(subprocess.run(["agentcore", "status"], capture_output=True, text=True).stdout or "torn down")

## Key takeaways

- **AgentCore Memory** gives a stateless agent **cross-session recall** as a managed service — no
  database to run, and `agentcore deploy` **auto-wires the IAM** and injects `MEMORY_<NAME>_ID`.
- **Two layers, two jobs:** short-term events give *immediate* cross-session recall; long-term
  extraction (SEMANTIC facts + USER_PREFERENCE) gives *distilled, learned* knowledge — but it's
  **asynchronous**, so design live demos around short-term recall.
- Memory layered on **without forking the agent**: `build_agent_options()` stayed the one source of
  truth; we injected recalled context through the additive `system_prompt_suffix` and recorded turns
  in the thin entrypoint.
- An **honest A/B** (memory on vs off, same deployment, same prompt) is how you prove the recall is
  real and not prompt phrasing.
- Records are **actor-scoped** via `users/{actorId}/…` namespaces — the seam you'd extend for
  multi-tenant isolation (out of scope here).

## Next steps

You've now climbed the full ladder: a local agent (M1), deployed (M2), observable (M4), and now with
memory (M3). A natural production follow-on is **multi-tenant memory** — deriving `actorId` from
inbound identity so each user gets an isolated namespace.
